In [1]:
import pandas as pd
DATA = ['data/sample_uxagent_runs_5_html_baseline.csv']
sims = pd.concat([pd.read_csv(d) for d in DATA])
sims

,starting_url,run_folder,persona_name,persona_description,persona_id,flow_name,flow_description,flow_id,category,name,text,text_len,input_tokens
0,https://www.barclays.co.uk/,runs/2025-12-01_14-59-28_6426,Young professional user,You are a young professional in your mid-20s. ...,0a232797-6024-44e4-ae13-6f3b375b10b5,Get customer support number,Get the number for customer service. Stop once...,d08c86b0-a015-46f4-9271-bcc6b02b8fd8,financial,Sim 73,{'persona': 'Young professional user You are a...,18747,6037
1,https://www.nike.com/gb/,runs/2025-12-01_15-13-11_e441,First-time user,You are a first-time user with no prior experi...,1e501ffd-b1bf-414b-9c7e-493ee3c539ca,Find ladies cotton trousers on sale,Find a pair of trousers or joggers that contai...,a8369faf-4b1b-4ebe-b690-2264d23467f9,fashion,Sim 18,{'persona': 'First-time user You are a first-t...,6579,1985
2,https://www.primark.com/,runs/2025-12-01_15-32-04_6b22,Experienced user,You are an experienced user who has used this ...,774d242c-3d09-4102-84e1-602f488f385f,Find men's trainers,Find a pair of trainers in men's size 10 in a ...,4307f7ef-acc1-41bd-8437-08b7f0ecccd8,fashion,Sim 31,{'persona': 'Experienced user You are an exper...,8871,2743
3,https://www.santander.co.uk/,runs/2025-12-01_15-53-53_20e5,Generic user,A generic user.,cb5ae8b6-d961-4455-b3e8-eba75c70cd25,Find a credit card,Find a credit card that suits your needs. Stop...,7b9f4a1b-15d1-42d3-a578-a71e4d428e9a,financial,Sim 141,"{'persona': 'Generic user A generic user.', 'i...",16429,4898
4,https://www.zara.com/uk/,runs/2025-12-01_14-44-15_8a34,Older user,You are a retiree in their 70s. You have typic...,4169e998-48c6-4697-98d1-0113b448a8c5,Find returns policy,Find the returns policy. Stop once you are sat...,233e9131-9020-4254-9157-50013424a695,fashion,Sim 56,{'persona': 'Older user You are a retiree in t...,10709,3549
5,https://www.nationwide.co.uk/,runs/2025-12-01_16-36-42_e360,First-time user,You are a first-time user with no prior experi...,1e501ffd-b1bf-414b-9c7e-493ee3c539ca,Locate a nearby branch,Find a branch that's near WC1E 6AE. Stop once ...,f0caf61d-07d0-40f5-9115-c383f7481a8f,financial,Sim 110,{'persona': 'First-time user You are a first-t...,15621,4600
6,https://www.hsbc.co.uk/,runs/2025-12-01_16-39-54_9fdc,First-time user,You are a first-time user with no prior experi...,1e501ffd-b1bf-414b-9c7e-493ee3c539ca,Find a credit card,Find a credit card that suits your needs. Stop...,7b9f4a1b-15d1-42d3-a578-a71e4d428e9a,financial,Sim 78,{'persona': 'First-time user You are a first-t...,21039,6621
7,https://www.next.co.uk/,runs/2025-12-01_16-50-45_edad,Young professional user,You are a young professional in your mid-20s. ...,0a232797-6024-44e4-ae13-6f3b375b10b5,Find ladies cotton trousers on sale,Find a pair of trousers or joggers that contai...,a8369faf-4b1b-4ebe-b690-2264d23467f9,fashion,Sim 12,{'persona': 'Young professional user You are a...,248751,88740
8,https://www.lloydsbank.com/,runs/2025-12-01_17-04-15_7a81,Young professional user,You are a young professional in your mid-20s. ...,0a232797-6024-44e4-ae13-6f3b375b10b5,Locate a nearby branch,Find a branch that's near WC1E 6AE. Stop once ...,f0caf61d-07d0-40f5-9115-c383f7481a8f,financial,Sim 104,{'persona': 'Young professional user You are a...,10321,3262
9,https://www.natwest.com/,runs/2025-12-01_17-13-33_a270,Young professional user,You are a young professional in your mid-20s. ...,0a232797-6024-44e4-ae13-6f3b375b10b5,Find a credit card,Find a credit card that suits your needs. Stop...,7b9f4a1b-15d1-42d3-a578-a71e4d428e9a,financial,Sim 132,{'persona': 'Young professional user You are a...,16558,5158


In [2]:
import json
import os
import anthropic

client = anthropic.Anthropic()

def count_tokens(text):
    response = client.messages.count_tokens(
    model="claude-sonnet-4-5",
    messages=[{
        "role": "user",
        "content": text
    }],
)

    return json.loads(response.model_dump_json())['input_tokens']

def get_text_from_run(run_folder):
    with open(run_folder + '/basic_info.json', 'r') as f:
        basic_info = json.load(f)

    # get the simplified HTML_0 from the run_folder/simp_html folder
    with open(run_folder + '/simp_html/simp_html_1.html', 'r') as f:
        simplified_html = f.read()

    output= str(basic_info) + '\n\n HOMEPAGE HTML \n ' + str(simplified_html)
    
    print(f'{run_folder}: {count_tokens(output)}')
    
    return output
    
sims['text'] = sims.apply(lambda x: get_text_from_run(x['run_folder']), axis=1)

sims['text_len'] = sims['text'].apply(lambda x: len(x))
sims['input_tokens'] = sims['text'].apply(lambda x: count_tokens(x))
# 
sims['contains_cookies'] = sims['text'].str.contains("cookies")

runs/2025-12-01_14-59-28_6426: 3077
runs/2025-12-01_15-13-11_e441: 5969
runs/2025-12-01_15-32-04_6b22: 2515
runs/2025-12-01_15-53-53_20e5: 4802
runs/2025-12-01_14-44-15_8a34: 3745
runs/2025-12-01_16-36-42_e360: 4238
runs/2025-12-01_16-39-54_9fdc: 6621
runs/2025-12-01_16-50-45_edad: 88005
runs/2025-12-01_17-04-15_7a81: 2204
runs/2025-12-01_17-13-33_a270: 5158
runs/2025-12-03_15-36-10_0913: 4597
runs/2025-12-03_15-42-09_cad1: 6230
runs/2025-12-03_15-45-50_684b: 3013
runs/2025-12-03_15-53-29_aaf1: 2828
runs/2025-12-03_16-00-31_3092: 6583
runs/2025-12-03_16-24-57_495f: 2503
runs/2025-12-03_16-55-29_75de: 88695
runs/2025-12-03_17-45-04_25d8: 5990
runs/2025-12-03_18-01-45_9b47: 7419


KeyboardInterrupt: 

In [ ]:
sims['contains_cookies'].mean()

np.float64(0.85)

In [ ]:
sims.iloc[0]['text']

'{\'persona\': \'Young professional user You are a young professional in your mid-20s. You have typical digital skills for your age.\', \'intent\': \'Get customer support number Get the number for customer service. Stop once you are satisfied that you have found the customer support number.\'}\n\n HOMEPAGE HTML \n <html><body id="page-5f39432f7d" parser-is-focused="true"><div id="page-body"><div id="pers-header" role="banner"><span id="pers-logo"><a aria-label="Barclays homepage" parser-semantic-id="barclays_home" parser-clickable="true"><span>Barclays Home</span><span aria-hidden="true"><img><img alt="Barclays"></span></a></span><button id="dropdown-button-fa678dccb9" aria-haspopup="menu" parser-semantic-id="personal" parser-clickable="true"><span><span>Personal</span></span></button><div><a id="PER_Util_Contact" title="Contact us" parser-semantic-id="contact_us" parser-clickable="true"><span>Contact us</span></a><a id="PER_Util_Finder" title="Find Barclays" parser-semantic-id="find_b

In [ ]:
sims.to_csv('data/sample_uxagent_runs_5_html_baseline.csv', index=False)